## Construct quadratic program

In [1]:
from qiskit_optimization.applications import Maxcut
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import NumPyMinimumEigensolver
from qiskit_algorithms.optimizers import NELDER_MEAD

import numpy as np
import networkx as nx

n = 10
G = nx.random_regular_graph(3, n, seed=123)
max_cut = Maxcut(G)
qp_max_cut = max_cut.to_quadratic_program()
print(qp_max_cut.prettyprint())

Problem name: Max-cut

Maximize
  -2*x_0*x_1 - 2*x_0*x_3 - 2*x_0*x_8 - 2*x_1*x_8 - 2*x_1*x_9 - 2*x_2*x_3
  - 2*x_2*x_7 - 2*x_2*x_8 - 2*x_3*x_6 - 2*x_4*x_5 - 2*x_4*x_6 - 2*x_4*x_7
  - 2*x_5*x_7 - 2*x_5*x_9 - 2*x_6*x_9 + 3*x_0 + 3*x_1 + 3*x_2 + 3*x_3 + 3*x_4
  + 3*x_5 + 3*x_6 + 3*x_7 + 3*x_8 + 3*x_9

Subject to
  No constraints

  Binary variables (10)
    x_0 x_1 x_2 x_3 x_4 x_5 x_6 x_7 x_8 x_9



In [2]:
exact = MinimumEigenOptimizer(NumPyMinimumEigensolver())
result_max_cut = exact.solve(qp_max_cut)
print(result_max_cut.prettyprint())

objective function value: 13.0
variable values: x_0=0.0, x_1=0.0, x_2=0.0, x_3=1.0, x_4=1.0, x_5=0.0, x_6=0.0, x_7=1.0, x_8=1.0, x_9=1.0
status: SUCCESS


## Solve using LCC

In [8]:
from lcc import LCCVQE

solver = LCCVQE(
    qp_max_cut,
    optimizer=NELDER_MEAD(),
)

params = np.random.rand(2 * qp_max_cut.get_num_vars()) * 2 * np.pi
print('Initial params:', params.view())
solver.compute_energy(params)

Initial params: [0.42535372 2.98451949 2.18602021 2.48452122 6.08360035 1.96302787
 4.56366436 2.29341738 3.0160942  4.16488002 2.11799644 2.68414737
 0.14126072 4.00712128 4.46946023 3.69332855 2.97493029 4.109932
 5.8489188  3.94900509]


-0.380859375

In [9]:
result = solver.solve(params)
print(result)

{   'fun': -3.487548828125,
    'jac': None,
    'nfev': 1000,
    'nit': 500,
    'njev': None,
    'x': array([0.30109789, 3.14808353, 2.65907676, 2.03646212, 4.26407126,
       2.98067029, 6.04879824, 2.97914924, 1.44662283, 4.06075405,
       2.98894824, 3.14620729, 0.15038979, 5.28570048, 3.06475078,
       3.27098786, 2.75806753, 3.64746687, 4.33814919, 3.32628313])}


In [10]:
print('Expectation:', -(result.fun + solver.offset))

Expectation: 10.987548828125


In [6]:
solver.offset

-7.5